# Use case — `Utility/time_delay_interpolation.py`

Classical two-curve baseline using three linear-interpolation comparisons and their symmetric mean.

**Convention:** `t_B_shifted = t_B - delay`. A positive delay means B is observed later than A. Start with the `quick` profile or the explicit parameters below, then inspect every score profile before running Monte Carlo uncertainty.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

from Utility import time_delay_interpolation as td

## Input identifiers

Use strings for Gaia IDs to prevent accidental floating-point rounding.

In [ ]:
INPUT_CSV = PROJECT_ROOT / "data" / "cleaned_lightcurves.csv"  # preprocessed light curves.
SOURCE_ID = "Source_ID"                               # System containing both light curves.
COMPONENT_A = "Component_A_ID"                             # Reference component; its delay is fixed to zero.
COMPONENT_B = "Component_B_ID"                             # Component whose delay relative to A is estimated.

if not INPUT_CSV.exists():
    raise FileNotFoundError(f"Supply the cleaned canonical CSV first: {INPUT_CSV}")

## Estimator hyperparameters

The dictionary below reproduces the original notebook adaptation. Equivalent named profiles are available in `configs/time_delay_profiles.json`.

In [ ]:
# ============================================================
# LINEAR-INTERPOLATION TIME-DELAY CONFIGURATION
# ============================================================
#
# This method estimates the time delay between two light curves,
# called A and B.
#
# Curve A is the reference curve. The program shifts the observation
# times of curve B using:
#
#     shifted_time_B = time_B - delay
#
# A positive delay means that B was observed later than A.
#
# For a first run, it is recommended to keep these values.
# The parameters most commonly changed are dmin, dmax, ngrid,
# min_points, min_frac, and min_span_frac.


estimator_kwargs = {

    # --------------------------------------------------------
    # 1. RANGE IN WHICH THE DELAY IS SEARCHED
    # --------------------------------------------------------

    # Smallest B-minus-A delay that the program is allowed to test.
    # Here, the search starts at -500 days.
    #
    # Decrease this value, for example to -700, if a larger
    # negative delay is scientifically possible.
    "dmin": -500,

    # Largest B-minus-A delay that the program is allowed to test.
    # Here, the search stops at +500 days.
    #
    # Increase this value if a delay greater than 500 days
    # is scientifically possible.
    "dmax": 500,

    # Number of delay values tested between dmin and dmax during
    # the initial search.
    #
    # A larger value gives a more detailed search, but increases
    # computation time.
    #
    # With dmin=-500, dmax=500, and ngrid=400, the initial spacing is:
    #
    #     (500 - (-500)) / (400 - 1) ≈ 2.51 days
    #
    # The program can later improve this precision when refine=True.
    "ngrid": 400,


    # --------------------------------------------------------
    # 2. INTERPOLATION SETTINGS
    # --------------------------------------------------------

    # Number of regularly spaced points used when both light curves
    # are interpolated onto the same common time grid.
    #
    # This parameter is used only for the comparison:
    #
    #     interpolated A versus interpolated B
    #
    # It does not change the two comparisons that use observations
    # from one of the original light curves.
    #
    # Increasing this value produces a denser common grid, but the
    # interpolated points are not independent observations.
    "common_grid_size": 200,

    # True allows the program to fit a constant vertical flux difference
    # between the two light curves.
    #
    # The comparison becomes approximately:
    #
    #     flux_A ≈ flux_B + constant_offset
    #
    # This is useful when A and B have different average flux levels.
    #
    # False forces the constant offset to zero.
    #
    # This parameter corrects only a constant difference. It does not
    # model a time-dependent microlensing trend.
    "fit_offset": True,


    # --------------------------------------------------------
    # 3. MEASUREMENT-ERROR WEIGHTS
    # --------------------------------------------------------

    # Smallest measurement error allowed after each light curve has
    # been normalised.
    #
    # The program approximately uses:
    #
    #     normalised_error = max(
    #         flux_error / robust_flux_scale,
    #         sigma_floor,
    #     )
    #
    # The robust flux scale is calculated from the median absolute
    # deviation of the flux values.
    #
    # This lower limit prevents an observation with an extremely small
    # reported error from dominating the entire comparison.
    #
    # Increasing sigma_floor makes the weights more similar.
    # Decreasing it gives more importance to measurements reported
    # with very small errors.
    "sigma_floor": 0.05,


    # --------------------------------------------------------
    # 4. REQUIRED TEMPORAL OVERLAP
    # --------------------------------------------------------

    # Minimum number of original observations that each light curve
    # must retain inside the common time interval after B is shifted.
    #
    # If A or B has fewer than 10 usable observations in the overlap,
    # the tested delay is automatically rejected.
    #
    # This counts original observations, not artificially created
    # interpolation-grid points.
    "min_points": 10,

    # Minimum fraction of observations that must remain available
    # in each light curve after the time shift.
    #
    # 0.50 means that at least 50% of the observations from A and
    # at least 50% of the observations from B must remain in the
    # common time interval.
    #
    # If even one curve falls below this value, the tested delay
    # is rejected.
    "min_frac": 0.50,

    # Minimum fraction of the original temporal duration that must
    # remain inside the common time interval.
    #
    # 0.50 means that the overlap must cover at least 50% of the
    # duration of the shorter light curve.
    #
    # For two light curves spanning approximately 10 years, this
    # requires approximately 5 years of common temporal coverage.
    #
    # This condition is different from min_frac: many observations
    # concentrated in a short period may satisfy min_frac without
    # satisfying min_span_frac.
    "min_span_frac": 0.50,


    # --------------------------------------------------------
    # 5. OPTIONAL PENALTY FOR REDUCED OVERLAP
    # --------------------------------------------------------

    # Additional cost applied to delays that retain less temporal
    # coverage or fewer observations.
    #
    # 0.0 disables this optional penalty. The mandatory min_points,
    # min_frac, and min_span_frac conditions remain active.
    #
    # A positive value can discourage solutions based on a relatively
    # short overlap, but it can also move the selected delay away from
    # the true value if the penalty is too strong.
    #
    # Therefore, keep 0.0 unless the effect of the penalty has been
    # validated on simulated data with known delays.
    "overlap_penalty": 0.0,


    # --------------------------------------------------------
    # 6. LOCAL REFINEMENT OF THE BEST DELAY
    # --------------------------------------------------------

    # True asks the program to refine the best delay found on the
    # initial grid using a continuous local optimiser.
    #
    # This allows the final result to fall between two values of
    # the initial delay grid.
    #
    # False returns only one of the delays explicitly present in
    # the initial grid.
    "refine": True,

    # Numerical convergence tolerance of the local optimiser, in days.
    #
    # 0.05 days corresponds to approximately 1.2 hours.
    #
    # A smaller value requests a more precise numerical search but may
    # require more calculations. It does not make the observations more
    # informative and does not represent the scientific uncertainty.
    #
    # For example, scalar_xatol=0.05 does not mean that the final delay
    # has an uncertainty of ±0.05 days. Scientific uncertainty must be
    # estimated separately, for example with Monte Carlo simulations.
    "scalar_xatol": 0.05,


    # --------------------------------------------------------
    # 7. DISPLAY OF THE RESULTS
    # --------------------------------------------------------

    # True displays the estimated delays, scores, overlap information,
    # and other useful diagnostic values.
    #
    # The program reports four results:
    #
    # 1. original A compared with interpolated B;
    # 2. interpolated A compared with original B;
    # 3. interpolated A compared with interpolated B;
    # 4. the average score from the three comparisons.
    #
    # False performs the same calculations without printing details.
    "verbose": True,
}

MC_SAMPLES = 300              # Independent flux-error draws; use 20 for a test and >=300 for final work.
MC_RANDOM_SEED = 42           # Makes uncertainty samples reproducible.
MC_ERROR_SCALE = 1.0          # 1.0 uses flux_obs_error exactly; 2.0 doubles every measurement sigma.
MC_PROGRESS_EVERY = 25        # Print progress every N draws; set 0 to suppress progress lines.

In [ ]:
df = td.load_lightcurve_csv(INPUT_CSV)
system = td.get_pair_from_df(
    df,
    source_id=SOURCE_ID,
    comp_a=COMPONENT_A,
    comp_b=COMPONENT_B,
    names=("A reference", "B"),
)

result = td.estimate_time_delay_linear_interpolation(
    system["curves"],
    **estimator_kwargs,
)
td.plot_delay_profiles_interpolation(result)
td.plot_best_alignments_interpolation(result)

## Measurement-error uncertainty

This is Monte Carlo propagation of `flux_obs_error`, not MCMC sampling of a delay posterior. Inspect histograms for multiple modes.

In [ ]:
uncertainty = td.run_fluxobs_error_mc_interpolation(
    system=system,
    estimator_kwargs=estimator_kwargs,
    n_samples=MC_SAMPLES,
    random_seed=MC_RANDOM_SEED,
    error_scale=MC_ERROR_SCALE,
    base_res=result,
    progress_every=MC_PROGRESS_EVERY,
    verbose=True,
)
td.print_fluxobs_error_mc_interpolation(uncertainty)
td.plot_fluxobs_error_mc_interpolation(uncertainty, bins=30)
display(uncertainty["summary"])

## Command-line equivalent

```bash
python -m Utility.time_delay_interpolation data/cleaned_lightcurves.csv --source-id 3361094865862486656 --component-a 3361094865862486721 --component-b 3361094865862486723 --profile legacy_notebook --mc-samples 300
```